In [3]:
from pyspark.sql import SparkSession
import boto3

minio_endpoint = "http://172.16.0.4:9000"
spark = (
    SparkSession.builder
    .appName("MinIOExample")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    .config("spark.hadoop.fs.s3a.endpoint", "http://172.16.0.4:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

In [4]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

raw_bucket = "s3a://bronze/datalake/raw/transactions/"

# Gerando 100 mil linhas base com funções nativas do Spark
df_base = spark.range(1, 100001).select(
    F.col("id").cast(IntegerType()).alias("transaction_id"),
    F.date_format(F.date_add(F.lit("2026-01-01"), (F.rand() * 300).cast("int")), "yyyy-MM-dd").alias("tx_date"),
    F.element_at(F.array(F.lit("SP"), F.lit("RJ"), F.lit("MG"), F.lit("ES"), F.lit("PR"), F.lit("XX")), (F.rand() * 6 + 1).cast("int")).alias("state"),
    (F.rand() * 5000 - 50).alias("amount"), # Introduz valores negativos aleatoriamente
    F.element_at(F.array(F.lit("ACTIVE"), F.lit("PENDING"), F.lit("CANCELLED"), F.lit("CORRUPT")), (F.rand() * 4 + 1).cast("int")).alias("status")
)

# Injetando problemas deliberados de qualidade para o laboratório:
# 1. Nulos em 5% dos IDs
# 2. Duplicações forçadas de registros
df_anomalies = df_base \
    .withColumn("transaction_id", F.when(F.rand() < 0.05, None).otherwise(F.col("transaction_id")))

# Duplica 2% dos dados aleatoriamente
df_raw_final = df_anomalies.unionByName(df_anomalies.sample(withReplacement=True, fraction=0.02, seed=42))

print(f"Gravando dados no Datalake {raw_bucket}")
# Salvando os dados brutos no MinIO (Camada Bronze)
df_raw_final.write.mode("overwrite").option("header", "true").csv(raw_bucket)

print(f"Total de registros na Raw: {df_raw_final.count():,}")

Gravando dados no Datalake s3a://bronze/datalake/raw/transactions/
Total de registros na Raw: 102,065
